# Save verified original-itemized STL as an isolated Gold candidate

**This notebook creates one new release-specific Delta candidate table (202 records).** It does not update source Gold, approvals, live dashboard readers or model pointers. An identical rerun verifies existing data without rewriting; differing existing data stops.

The reviewed calendar/components are bundled in this notebook, checked against current pinned Gold inputs before any write. It does not fit a model or require statsmodels, CSV uploads or helper files. This is retrospective diagnostic STL, not a forecast or verified SKU demand.


In [ ]:
import base64

import hashlib

import json

import math

import re

import zlib

from decimal import Decimal, InvalidOperation

SOURCE_TABLE = 'workspace.medshield_original_gold_20261026.original_fd95b9e3abab028c8e41_itemized_candidates'

RECEIPT_TABLE = 'workspace.medshield_original_gold_20261026.original_fd95b9e3abab028c8e41_receipt'

DATASET_ID = 'e87cae21986db32bcade298d69e652236be9db65f2685db51355003f1439ff10'

POLICY = 'original_source_layers_v1_candidate_only'

RECEIPT_SHA256 = 'd69414d3bb72c706bba30f24489074e4677f403277e18c64bb8d2e643b0012dd'

SOURCE_KEYS_SHA256 = 'da74cc50c67e22fd33928f5ca3faaebc3e21a6e1b76f7f169829bf6515d8c580'

MONTHLY_LOGICAL_SHA256 = '9d087afd9dce2ee41da1701d34cbb0083d92d5b5b24dc8aac6471d83e5ed8d0e'

MONTHLY_FIELDS = ['month', 'source_records', 'net_eligible_records', 'net_sales_php',
                  'supported_gross_profit_php', 'matching_margin_revenue_php']

EXPECTED_TOTALS = {'source_records': '40171', 'net_eligible_records': '39286',
                   'net_sales_php': '638015351.530000',
                   'supported_gross_profit_php': '334666271.550000',
                   'matching_margin_revenue_php': '566954921.920000'}

def require(condition, message):
    if not condition:
        raise ValueError(message)

def numeric(value):
    require(value is not None and not isinstance(value, bool), 'Missing/boolean monthly value')
    try:
        result = Decimal(str(value))
    except InvalidOperation as error:
        raise ValueError('Invalid monthly number') from error
    require(result.is_finite(), 'Nonfinite monthly value')
    return result

def month_index(period):
    require(isinstance(period, str) and re.fullmatch(r'\d{4}-(0[1-9]|1[0-2])', period), 'Invalid calendar month')
    year, month = map(int, period.split('-'))
    return year * 12 + month - 1

def monthly_identity(rows):
    require(0 < len(rows) <= 2400, 'Bounded monthly input required')
    ordered = sorted(rows, key=lambda row: month_index(row['month']))
    require(len({row['month'] for row in ordered}) == len(ordered), 'Duplicate calendar months')
    normalized = []
    for row in ordered:
        record = [row['month']]
        for field in MONTHLY_FIELDS[1:]:
            value = numeric(row[field])
            if field.endswith('records'):
                require(value == int(value) and value >= 0, 'Invalid record count')
                record.append(str(int(value)))
            else:
                require(value == value.quantize(Decimal('.000001')), 'Unsupported decimal precision')
                record.append(format(value, '.6f'))
        require(numeric(row['source_records']) > 0, 'An observed month must have source records')
        require(numeric(row['net_eligible_records']) <= numeric(row['source_records']), 'Eligibility exceeds records')
        normalized.append(record)
    payload = json.dumps(normalized, separators=(',', ':'), ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def validate_monthly(rows):
    require(monthly_identity(rows) == MONTHLY_LOGICAL_SHA256, 'Monthly input differs from the reviewed export; stop and reassess')
    for field, expected in EXPECTED_TOTALS.items():
        require(sum((numeric(row[field]) for row in rows), Decimal(0)) == Decimal(expected), 'Monthly total differs: ' + field)
    return sorted(rows, key=lambda row: row['month'])

def calendar_rows(rows):
    observed = {row['month']: row for row in rows}
    first, last = min(observed), max(observed)
    start = int(first[:4]) * 12
    stop = (int(last[:4]) + 1) * 12
    require(stop - start <= 2400, 'Calendar safety bound exceeded')
    result = []
    for index in range(start, stop):
        period = f'{index // 12:04d}-{index % 12 + 1:02d}'
        row = observed.get(period)
        result.append({'period': period,
                       'net_sales_php': str(row['net_sales_php']) if row else None,
                       'status': 'observed_source_month' if row else
                           'outside_observed_source_range' if period < first or period > last else 'missing_source_month'})
    return result

def read_gold_months(session):
    for table in (SOURCE_TABLE, RECEIPT_TABLE):
        history = session.sql(f'DESCRIBE HISTORY {table} LIMIT 1').collect()
        require(len(history) == 1 and int(history[0]['version']) == 0, 'Pinned Gold table changed after review')
    receipt = session.sql(f"SELECT dataset_id, payload_sha256, payload_b64 FROM {RECEIPT_TABLE} VERSION AS OF 0 WHERE record_id = 'gold_receipt'").collect()
    require(len(receipt) == 1 and receipt[0]['dataset_id'] == DATASET_ID and receipt[0]['payload_sha256'] == RECEIPT_SHA256, 'Gold receipt identity differs')
    payload = zlib.decompress(base64.b64decode(receipt[0]['payload_b64'], validate=True))
    require(hashlib.sha256(payload).hexdigest() == RECEIPT_SHA256, 'Gold receipt bytes differ')
    checks = session.sql(f"""SELECT COUNT(*) AS records, COUNT(DISTINCT source_record_id) AS unique_records,
      SHA2(CONCAT_WS('|', SORT_ARRAY(COLLECT_LIST(source_record_id))), 256) AS keys_sha256,
      COUNT_IF(dataset_id IS NULL OR dataset_id <> '{DATASET_ID}' OR policy_version IS NULL
        OR policy_version <> '{POLICY}' OR record_kind IS NULL OR record_kind <> 'itemized_sale'
        OR date_delivered IS NULL OR product_key IS NULL OR normalized_product IS NULL
        OR category_status IS NULL OR category_status <> 'approved'
        OR product_category IS NULL OR product_category NOT IN ('medicine','medical_supply','medical_equipment','non_medical','unclassified')
        OR is_net_sales_technically_eligible IS NULL
        OR (is_net_sales_technically_eligible AND net_sales IS NULL)
        OR is_gross_margin_technically_eligible IS NULL
        OR (is_gross_margin_technically_eligible AND (NOT is_net_sales_technically_eligible OR gross_margin_amount IS NULL))
        OR NOT (forecast_eligible <=> FALSE) OR NOT (source_unit_approved <=> FALSE)
        OR NOT (financial_release_approved <=> FALSE)) AS invalid_rows
      FROM {SOURCE_TABLE} VERSION AS OF 0""").collect()
    require(len(checks) == 1, 'Gold controls are missing')
    check = checks[0]
    require(check['records'] == 40171 and check['unique_records'] == 40171
            and check['keys_sha256'] == SOURCE_KEYS_SHA256 and check['invalid_rows'] == 0, 'Gold source/keys/eligibility/approval controls failed')
    rows = session.sql(f"""SELECT DATE_FORMAT(date_delivered, 'yyyy-MM') AS month,
      COUNT(*) AS source_records, COUNT_IF(is_net_sales_technically_eligible) AS net_eligible_records,
      SUM(CASE WHEN is_net_sales_technically_eligible THEN net_sales END) AS net_sales_php,
      SUM(CASE WHEN is_gross_margin_technically_eligible THEN gross_margin_amount END) AS supported_gross_profit_php,
      SUM(CASE WHEN is_gross_margin_technically_eligible THEN net_sales END) AS matching_margin_revenue_php
      FROM {SOURCE_TABLE} VERSION AS OF 0 WHERE record_kind = 'itemized_sale'
      GROUP BY DATE_FORMAT(date_delivered, 'yyyy-MM') ORDER BY month""").collect()
    return validate_monthly([row.asDict() for row in rows])


## Checked result package and immutable candidate writer
The package binds the supplied remote exports, monthly input and exact source/method identity.


In [ ]:
PACKAGE = json.loads('{"policy":"original_itemized_stl_export_candidate_v1","source":{"source_table":"workspace.medshield_original_gold_20261026.original_fd95b9e3abab028c8e41_itemized_candidates","source_delta_version":0,"dataset_id":"e87cae21986db32bcade298d69e652236be9db65f2685db51355003f1439ff10","policy":"original_source_layers_v1_candidate_only","receipt_sha256":"d69414d3bb72c706bba30f24489074e4677f403277e18c64bb8d2e643b0012dd","monthly_sha256":"9d087afd9dce2ee41da1701d34cbb0083d92d5b5b24dc8aac6471d83e5ed8d0e"},"method":{"name":"STL","period":12,"seasonal_window":13,"robust":true,"minimum_contiguous_months":36,"missing_policy":"no_imputation_split_at_gaps","statsmodels_version":"0.14.6","numpy_version":"2.3.4"},"source_export_hashes":{"calendar":"dd38038f620a8aa3fec09aa2d40163ec1e7dbef717e405c02dc867a0941951fd","components":"d47763d7a195a4dad1d731869ba18b369e90b03195025e2536651a1baf4328e8"},"monthly_rows":[{"month":"2017-01","source_records":"515","net_eligible_records":"512","net_sales_php":"2288688.880000","supported_gross_profit_php":"1563511.660000","matching_margin_revenue_php":"2288688.880000"},{"month":"2017-02","source_records":"454","net_eligible_records":"454","net_sales_php":"2478761.200000","supported_gross_profit_php":"1718333.260000","matching_margin_revenue_php":"2471021.200000"},{"month":"2017-03","source_records":"589","net_eligible_records":"557","net_sales_php":"2697198.470000","supported_gross_profit_php":"1638244.970000","matching_margin_revenue_php":"2362968.820000"},{"month":"2017-04","source_records":"401","net_eligible_records":"352","net_sales_php":"2655938.800000","supported_gross_profit_php":"1331733.890000","matching_margin_revenue_php":"1999347.200000"},{"month":"2017-09","source_records":"1","net_eligible_records":"1","net_sales_php":"480.000000","supported_gross_profit_php":"325.000000","matching_margin_revenue_php":"480.000000"},{"month":"2017-11","source_records":"9","net_eligible_records":"9","net_sales_php":"86200.000000","supported_gross_profit_php":"61288.900000","matching_margin_revenue_php":"86200.000000"},{"month":"2017-12","source_records":"9","net_eligible_records":"1","net_sales_php":"3900.000000","supported_gross_profit_php":"2720.000000","matching_margin_revenue_php":"3900.000000"},{"month":"2018-03","source_records":"418","net_eligible_records":"416","net_sales_php":"3457920.700000","supported_gross_profit_php":"2090495.000000","matching_margin_revenue_php":"3052738.600000"},{"month":"2018-04","source_records":"682","net_eligible_records":"682","net_sales_php":"6403396.280000","supported_gross_profit_php":"2827185.110000","matching_margin_revenue_php":"5018063.290000"},{"month":"2018-05","source_records":"634","net_eligible_records":"634","net_sales_php":"4591195.170000","supported_gross_profit_php":"2821895.010000","matching_margin_revenue_php":"4244749.170000"},{"month":"2018-06","source_records":"515","net_eligible_records":"515","net_sales_php":"4599086.640000","supported_gross_profit_php":"2661903.010000","matching_margin_revenue_php":"4017156.640000"},{"month":"2018-07","source_records":"587","net_eligible_records":"587","net_sales_php":"5034975.380000","supported_gross_profit_php":"2812895.330000","matching_margin_revenue_php":"4618875.940000"},{"month":"2018-08","source_records":"585","net_eligible_records":"585","net_sales_php":"4085611.820000","supported_gross_profit_php":"2320349.850000","matching_margin_revenue_php":"3728264.890000"},{"month":"2018-09","source_records":"684","net_eligible_records":"684","net_sales_php":"4066997.220000","supported_gross_profit_php":"2520488.660000","matching_margin_revenue_php":"4029876.510000"},{"month":"2018-10","source_records":"807","net_eligible_records":"807","net_sales_php":"7274326.830000","supported_gross_profit_php":"3796308.790000","matching_margin_revenue_php":"6313542.230000"},{"month":"2018-11","source_records":"845","net_eligible_records":"845","net_sales_php":"7571263.890000","supported_gross_profit_php":"4078448.140000","matching_margin_revenue_php":"6712233.740000"},{"month":"2018-12","source_records":"424","net_eligible_records":"424","net_sales_php":"4534917.340000","supported_gross_profit_php":"2398752.860000","matching_margin_revenue_php":"3146994.380000"},{"month":"2019-01","source_records":"737","net_eligible_records":"737","net_sales_php":"4459965.470000","supported_gross_profit_php":"1772751.680000","matching_margin_revenue_php":"2613721.840000"},{"month":"2019-02","source_records":"484","net_eligible_records":"482","net_sales_php":"5473644.670000","supported_gross_profit_php":"1570400.990000","matching_margin_revenue_php":"2470989.500000"},{"month":"2019-03","source_records":"539","net_eligible_records":"521","net_sales_php":"6995759.140000","supported_gross_profit_php":"2779093.550000","matching_margin_revenue_php":"4214764.930000"},{"month":"2019-04","source_records":"799","net_eligible_records":"789","net_sales_php":"9438917.290000","supported_gross_profit_php":"4594590.790000","matching_margin_revenue_php":"6370271.300000"},{"month":"2019-05","source_records":"646","net_eligible_records":"643","net_sales_php":"9568922.120000","supported_gross_profit_php":"4396497.160000","matching_margin_revenue_php":"6042074.490000"},{"month":"2019-06","source_records":"540","net_eligible_records":"539","net_sales_php":"6948030.480000","supported_gross_profit_php":"4022965.260000","matching_margin_revenue_php":"5408966.070000"},{"month":"2019-07","source_records":"647","net_eligible_records":"639","net_sales_php":"8694463.130000","supported_gross_profit_php":"4080324.510000","matching_margin_revenue_php":"5662574.040000"},{"month":"2019-08","source_records":"513","net_eligible_records":"510","net_sales_php":"5568251.580000","supported_gross_profit_php":"3146257.710000","matching_margin_revenue_php":"4486498.680000"},{"month":"2019-09","source_records":"541","net_eligible_records":"541","net_sales_php":"5394350.610000","supported_gross_profit_php":"2762588.300000","matching_margin_revenue_php":"3941420.570000"},{"month":"2020-01","source_records":"614","net_eligible_records":"611","net_sales_php":"7514695.050000","supported_gross_profit_php":"2604913.920000","matching_margin_revenue_php":"3740836.740000"},{"month":"2020-02","source_records":"459","net_eligible_records":"458","net_sales_php":"4691903.000000","supported_gross_profit_php":"2755650.900000","matching_margin_revenue_php":"3846729.410000"},{"month":"2020-03","source_records":"467","net_eligible_records":"464","net_sales_php":"6637589.080000","supported_gross_profit_php":"2918264.610000","matching_margin_revenue_php":"3955415.150000"},{"month":"2020-04","source_records":"207","net_eligible_records":"207","net_sales_php":"1525751.320000","supported_gross_profit_php":"897056.610000","matching_margin_revenue_php":"1321365.320000"},{"month":"2020-05","source_records":"418","net_eligible_records":"418","net_sales_php":"10486809.300000","supported_gross_profit_php":"5922379.270000","matching_margin_revenue_php":"8123767.020000"},{"month":"2020-06","source_records":"330","net_eligible_records":"330","net_sales_php":"4471548.750000","supported_gross_profit_php":"2241388.540000","matching_margin_revenue_php":"3126938.680000"},{"month":"2020-07","source_records":"170","net_eligible_records":"170","net_sales_php":"1553987.810000","supported_gross_profit_php":"844152.210000","matching_margin_revenue_php":"1342995.450000"},{"month":"2020-08","source_records":"269","net_eligible_records":"267","net_sales_php":"2983110.790000","supported_gross_profit_php":"1534325.490000","matching_margin_revenue_php":"2263338.750000"},{"month":"2020-09","source_records":"174","net_eligible_records":"174","net_sales_php":"2102633.270000","supported_gross_profit_php":"1127136.350000","matching_margin_revenue_php":"1924907.370000"},{"month":"2020-10","source_records":"421","net_eligible_records":"421","net_sales_php":"2469548.120000","supported_gross_profit_php":"1346486.910000","matching_margin_revenue_php":"1989440.030000"},{"month":"2020-11","source_records":"252","net_eligible_records":"252","net_sales_php":"4870933.120000","supported_gross_profit_php":"2471946.590000","matching_margin_revenue_php":"3853344.440000"},{"month":"2020-12","source_records":"196","net_eligible_records":"196","net_sales_php":"3771013.020000","supported_gross_profit_php":"2511317.280000","matching_margin_revenue_php":"3164804.730000"},{"month":"2021-01","source_records":"258","net_eligible_records":"258","net_sales_php":"5364380.250000","supported_gross_profit_php":"3487149.860000","matching_margin_revenue_php":"5359900.250000"},{"month":"2021-02","source_records":"213","net_eligible_records":"209","net_sales_php":"3682482.140000","supported_gross_profit_php":"2046824.440000","matching_margin_revenue_php":"3682482.140000"},{"month":"2021-03","source_records":"207","net_eligible_records":"206","net_sales_php":"2318360.720000","supported_gross_profit_php":"1553568.620000","matching_margin_revenue_php":"2318360.720000"},{"month":"2021-04","source_records":"258","net_eligible_records":"258","net_sales_php":"4928899.800000","supported_gross_profit_php":"3452714.960000","matching_margin_revenue_php":"4928899.800000"},{"month":"2021-05","source_records":"183","net_eligible_records":"183","net_sales_php":"2012916.830000","supported_gross_profit_php":"1119804.340000","matching_margin_revenue_php":"2002116.830000"},{"month":"2021-06","source_records":"253","net_eligible_records":"252","net_sales_php":"5225498.200000","supported_gross_profit_php":"3759325.120000","matching_margin_revenue_php":"5207433.200000"},{"month":"2021-07","source_records":"345","net_eligible_records":"345","net_sales_php":"9573660.330000","supported_gross_profit_php":"6844972.100000","matching_margin_revenue_php":"9567160.330000"},{"month":"2021-08","source_records":"305","net_eligible_records":"305","net_sales_php":"10533739.530000","supported_gross_profit_php":"5977103.000000","matching_margin_revenue_php":"10531039.530000"},{"month":"2021-09","source_records":"286","net_eligible_records":"286","net_sales_php":"8287059.600000","supported_gross_profit_php":"5263319.690000","matching_margin_revenue_php":"8287059.600000"},{"month":"2021-10","source_records":"381","net_eligible_records":"381","net_sales_php":"15279603.640000","supported_gross_profit_php":"9375684.820000","matching_margin_revenue_php":"15275859.640000"},{"month":"2021-11","source_records":"503","net_eligible_records":"503","net_sales_php":"6713804.040000","supported_gross_profit_php":"4791990.560000","matching_margin_revenue_php":"6704134.040000"},{"month":"2021-12","source_records":"215","net_eligible_records":"215","net_sales_php":"3939801.180000","supported_gross_profit_php":"2748313.420000","matching_margin_revenue_php":"3939801.180000"},{"month":"2022-01","source_records":"371","net_eligible_records":"371","net_sales_php":"5270833.750000","supported_gross_profit_php":"3837517.840000","matching_margin_revenue_php":"5080144.860000"},{"month":"2022-02","source_records":"453","net_eligible_records":"452","net_sales_php":"10568697.090000","supported_gross_profit_php":"6985336.620000","matching_margin_revenue_php":"10325195.750000"},{"month":"2022-03","source_records":"298","net_eligible_records":"298","net_sales_php":"5791537.630000","supported_gross_profit_php":"3940253.930000","matching_margin_revenue_php":"5391123.630000"},{"month":"2022-04","source_records":"314","net_eligible_records":"313","net_sales_php":"5638889.710000","supported_gross_profit_php":"3795460.110000","matching_margin_revenue_php":"5560660.220000"},{"month":"2022-05","source_records":"324","net_eligible_records":"324","net_sales_php":"3871579.020000","supported_gross_profit_php":"2316104.020000","matching_margin_revenue_php":"3520180.560000"},{"month":"2022-06","source_records":"489","net_eligible_records":"489","net_sales_php":"10350221.030000","supported_gross_profit_php":"7275105.290000","matching_margin_revenue_php":"10020070.160000"},{"month":"2022-07","source_records":"295","net_eligible_records":"295","net_sales_php":"3049922.430000","supported_gross_profit_php":"1830375.810000","matching_margin_revenue_php":"2798686.430000"},{"month":"2022-08","source_records":"194","net_eligible_records":"194","net_sales_php":"2138525.120000","supported_gross_profit_php":"1351331.160000","matching_margin_revenue_php":"2108727.120000"},{"month":"2022-09","source_records":"418","net_eligible_records":"417","net_sales_php":"6498970.540000","supported_gross_profit_php":"3677959.290000","matching_margin_revenue_php":"5717097.760000"},{"month":"2022-10","source_records":"296","net_eligible_records":"295","net_sales_php":"2788157.480000","supported_gross_profit_php":"1447522.730000","matching_margin_revenue_php":"2765788.480000"},{"month":"2022-11","source_records":"306","net_eligible_records":"300","net_sales_php":"3642191.630000","supported_gross_profit_php":"2076151.390000","matching_margin_revenue_php":"3582173.630000"},{"month":"2022-12","source_records":"222","net_eligible_records":"222","net_sales_php":"3625464.610000","supported_gross_profit_php":"2265930.870000","matching_margin_revenue_php":"3623372.610000"},{"month":"2023-01","source_records":"435","net_eligible_records":"423","net_sales_php":"7844551.200000","supported_gross_profit_php":"4582496.900000","matching_margin_revenue_php":"7738631.200000"},{"month":"2023-02","source_records":"753","net_eligible_records":"753","net_sales_php":"13794078.180000","supported_gross_profit_php":"6780024.200000","matching_margin_revenue_php":"13332610.180000"},{"month":"2023-03","source_records":"357","net_eligible_records":"357","net_sales_php":"2344746.440000","supported_gross_profit_php":"1328921.610000","matching_margin_revenue_php":"2295614.120000"},{"month":"2023-04","source_records":"215","net_eligible_records":"213","net_sales_php":"1559602.950000","supported_gross_profit_php":"775366.470000","matching_margin_revenue_php":"1371440.350000"},{"month":"2023-05","source_records":"625","net_eligible_records":"624","net_sales_php":"14537368.300000","supported_gross_profit_php":"6977451.490000","matching_margin_revenue_php":"13522718.300000"},{"month":"2023-06","source_records":"1455","net_eligible_records":"793","net_sales_php":"6910935.230000","supported_gross_profit_php":"3393523.660000","matching_margin_revenue_php":"6639428.180000"},{"month":"2023-07","source_records":"295","net_eligible_records":"278","net_sales_php":"2337078.500000","supported_gross_profit_php":"1151981.330000","matching_margin_revenue_php":"2285182.500000"},{"month":"2023-08","source_records":"348","net_eligible_records":"348","net_sales_php":"2588026.900000","supported_gross_profit_php":"1448354.770000","matching_margin_revenue_php":"2553709.660000"},{"month":"2023-09","source_records":"240","net_eligible_records":"235","net_sales_php":"5029660.600000","supported_gross_profit_php":"4183375.990000","matching_margin_revenue_php":"5029660.600000"},{"month":"2023-10","source_records":"243","net_eligible_records":"243","net_sales_php":"2411760.400000","supported_gross_profit_php":"1382968.720000","matching_margin_revenue_php":"2406560.400000"},{"month":"2023-11","source_records":"262","net_eligible_records":"245","net_sales_php":"1975407.650000","supported_gross_profit_php":"1046798.300000","matching_margin_revenue_php":"1843097.650000"},{"month":"2023-12","source_records":"153","net_eligible_records":"153","net_sales_php":"895750.500000","supported_gross_profit_php":"534319.040000","matching_margin_revenue_php":"804190.500000"},{"month":"2024-01","source_records":"271","net_eligible_records":"271","net_sales_php":"2124201.870000","supported_gross_profit_php":"1434941.770000","matching_margin_revenue_php":"2124201.870000"},{"month":"2024-02","source_records":"167","net_eligible_records":"167","net_sales_php":"1079764.410000","supported_gross_profit_php":"712331.090000","matching_margin_revenue_php":"1078594.410000"},{"month":"2024-03","source_records":"371","net_eligible_records":"371","net_sales_php":"2060009.790000","supported_gross_profit_php":"1420928.010000","matching_margin_revenue_php":"2058530.470000"},{"month":"2024-04","source_records":"179","net_eligible_records":"179","net_sales_php":"1929517.020000","supported_gross_profit_php":"949426.810000","matching_margin_revenue_php":"1902786.020000"},{"month":"2024-05","source_records":"232","net_eligible_records":"232","net_sales_php":"1848212.320000","supported_gross_profit_php":"1292967.410000","matching_margin_revenue_php":"1848212.320000"},{"month":"2024-06","source_records":"234","net_eligible_records":"234","net_sales_php":"6172208.160000","supported_gross_profit_php":"2980377.720000","matching_margin_revenue_php":"6061897.160000"},{"month":"2024-07","source_records":"247","net_eligible_records":"247","net_sales_php":"2519509.110000","supported_gross_profit_php":"1365655.840000","matching_margin_revenue_php":"2486999.110000"},{"month":"2024-08","source_records":"248","net_eligible_records":"248","net_sales_php":"2735618.080000","supported_gross_profit_php":"1440329.170000","matching_margin_revenue_php":"2735618.080000"},{"month":"2024-09","source_records":"196","net_eligible_records":"196","net_sales_php":"1935311.570000","supported_gross_profit_php":"1107266.100000","matching_margin_revenue_php":"1844411.570000"},{"month":"2024-10","source_records":"228","net_eligible_records":"228","net_sales_php":"1405131.230000","supported_gross_profit_php":"895730.460000","matching_margin_revenue_php":"1405131.230000"},{"month":"2024-11","source_records":"159","net_eligible_records":"159","net_sales_php":"1088480.000000","supported_gross_profit_php":"727280.500000","matching_margin_revenue_php":"1072280.000000"},{"month":"2024-12","source_records":"175","net_eligible_records":"175","net_sales_php":"3570738.160000","supported_gross_profit_php":"1610965.220000","matching_margin_revenue_php":"3549598.160000"},{"month":"2025-01","source_records":"215","net_eligible_records":"215","net_sales_php":"2737512.180000","supported_gross_profit_php":"1372735.440000","matching_margin_revenue_php":"2699812.180000"},{"month":"2025-02","source_records":"329","net_eligible_records":"329","net_sales_php":"8895702.320000","supported_gross_profit_php":"4127111.730000","matching_margin_revenue_php":"8798583.320000"},{"month":"2025-03","source_records":"231","net_eligible_records":"231","net_sales_php":"1982968.500000","supported_gross_profit_php":"981794.840000","matching_margin_revenue_php":"1970008.500000"},{"month":"2025-04","source_records":"186","net_eligible_records":"186","net_sales_php":"4439483.870000","supported_gross_profit_php":"2374438.000000","matching_margin_revenue_php":"4437642.870000"},{"month":"2025-05","source_records":"186","net_eligible_records":"186","net_sales_php":"7999318.650000","supported_gross_profit_php":"2793673.200000","matching_margin_revenue_php":"5423068.650000"},{"month":"2025-06","source_records":"473","net_eligible_records":"473","net_sales_php":"12502958.230000","supported_gross_profit_php":"6383331.860000","matching_margin_revenue_php":"12312808.130000"},{"month":"2025-07","source_records":"216","net_eligible_records":"216","net_sales_php":"3400386.810000","supported_gross_profit_php":"1860565.730000","matching_margin_revenue_php":"3379596.810000"},{"month":"2025-08","source_records":"209","net_eligible_records":"209","net_sales_php":"6359262.810000","supported_gross_profit_php":"3231884.560000","matching_margin_revenue_php":"6342982.810000"},{"month":"2025-09","source_records":"477","net_eligible_records":"477","net_sales_php":"16182279.770000","supported_gross_profit_php":"6873357.240000","matching_margin_revenue_php":"14798834.370000"},{"month":"2025-10","source_records":"289","net_eligible_records":"289","net_sales_php":"7489003.910000","supported_gross_profit_php":"2223913.810000","matching_margin_revenue_php":"4571844.910000"},{"month":"2025-11","source_records":"314","net_eligible_records":"314","net_sales_php":"6207648.450000","supported_gross_profit_php":"3050932.650000","matching_margin_revenue_php":"6196668.450000"},{"month":"2025-12","source_records":"662","net_eligible_records":"662","net_sales_php":"10373872.750000","supported_gross_profit_php":"4224684.740000","matching_margin_revenue_php":"9079498.670000"},{"month":"2026-01","source_records":"167","net_eligible_records":"167","net_sales_php":"2580952.710000","supported_gross_profit_php":"1428611.210000","matching_margin_revenue_php":"2480175.710000"},{"month":"2026-02","source_records":"415","net_eligible_records":"414","net_sales_php":"17872095.250000","supported_gross_profit_php":"9971626.030000","matching_margin_revenue_php":"17744079.250000"},{"month":"2026-03","source_records":"295","net_eligible_records":"293","net_sales_php":"8597855.260000","supported_gross_profit_php":"5264827.690000","matching_margin_revenue_php":"8505284.460000"},{"month":"2026-04","source_records":"356","net_eligible_records":"354","net_sales_php":"19158821.440000","supported_gross_profit_php":"9613676.430000","matching_margin_revenue_php":"19154779.090000"},{"month":"2026-05","source_records":"252","net_eligible_records":"252","net_sales_php":"9815789.610000","supported_gross_profit_php":"4927698.720000","matching_margin_revenue_php":"9815789.610000"},{"month":"2026-06","source_records":"443","net_eligible_records":"443","net_sales_php":"15028453.900000","supported_gross_profit_php":"7784215.860000","matching_margin_revenue_php":"14421457.180000"},{"month":"2026-07","source_records":"403","net_eligible_records":"403","net_sales_php":"24686328.780000","supported_gross_profit_php":"11703908.380000","matching_margin_revenue_php":"22603405.180000"},{"month":"2026-08","source_records":"377","net_eligible_records":"377","net_sales_php":"16404726.910000","supported_gross_profit_php":"2859666.980000","matching_margin_revenue_php":"10883275.950000"},{"month":"2026-09","source_records":"619","net_eligible_records":"619","net_sales_php":"26054378.810000","supported_gross_profit_php":"9924645.810000","matching_margin_revenue_php":"21086487.310000"}],"calendar":[{"period":"2017-01","net_sales_php":"2288688.880000","status":"observed_source_month"},{"period":"2017-02","net_sales_php":"2478761.200000","status":"observed_source_month"},{"period":"2017-03","net_sales_php":"2697198.470000","status":"observed_source_month"},{"period":"2017-04","net_sales_php":"2655938.800000","status":"observed_source_month"},{"period":"2017-05","net_sales_php":null,"status":"missing_source_month"},{"period":"2017-06","net_sales_php":null,"status":"missing_source_month"},{"period":"2017-07","net_sales_php":null,"status":"missing_source_month"},{"period":"2017-08","net_sales_php":null,"status":"missing_source_month"},{"period":"2017-09","net_sales_php":"480.000000","status":"observed_source_month"},{"period":"2017-10","net_sales_php":null,"status":"missing_source_month"},{"period":"2017-11","net_sales_php":"86200.000000","status":"observed_source_month"},{"period":"2017-12","net_sales_php":"3900.000000","status":"observed_source_month"},{"period":"2018-01","net_sales_php":null,"status":"missing_source_month"},{"period":"2018-02","net_sales_php":null,"status":"missing_source_month"},{"period":"2018-03","net_sales_php":"3457920.700000","status":"observed_source_month"},{"period":"2018-04","net_sales_php":"6403396.280000","status":"observed_source_month"},{"period":"2018-05","net_sales_php":"4591195.170000","status":"observed_source_month"},{"period":"2018-06","net_sales_php":"4599086.640000","status":"observed_source_month"},{"period":"2018-07","net_sales_php":"5034975.380000","status":"observed_source_month"},{"period":"2018-08","net_sales_php":"4085611.820000","status":"observed_source_month"},{"period":"2018-09","net_sales_php":"4066997.220000","status":"observed_source_month"},{"period":"2018-10","net_sales_php":"7274326.830000","status":"observed_source_month"},{"period":"2018-11","net_sales_php":"7571263.890000","status":"observed_source_month"},{"period":"2018-12","net_sales_php":"4534917.340000","status":"observed_source_month"},{"period":"2019-01","net_sales_php":"4459965.470000","status":"observed_source_month"},{"period":"2019-02","net_sales_php":"5473644.670000","status":"observed_source_month"},{"period":"2019-03","net_sales_php":"6995759.140000","status":"observed_source_month"},{"period":"2019-04","net_sales_php":"9438917.290000","status":"observed_source_month"},{"period":"2019-05","net_sales_php":"9568922.120000","status":"observed_source_month"},{"period":"2019-06","net_sales_php":"6948030.480000","status":"observed_source_month"},{"period":"2019-07","net_sales_php":"8694463.130000","status":"observed_source_month"},{"period":"2019-08","net_sales_php":"5568251.580000","status":"observed_source_month"},{"period":"2019-09","net_sales_php":"5394350.610000","status":"observed_source_month"},{"period":"2019-10","net_sales_php":null,"status":"missing_source_month"},{"period":"2019-11","net_sales_php":null,"status":"missing_source_month"},{"period":"2019-12","net_sales_php":null,"status":"missing_source_month"},{"period":"2020-01","net_sales_php":"7514695.050000","status":"observed_source_month"},{"period":"2020-02","net_sales_php":"4691903.000000","status":"observed_source_month"},{"period":"2020-03","net_sales_php":"6637589.080000","status":"observed_source_month"},{"period":"2020-04","net_sales_php":"1525751.320000","status":"observed_source_month"},{"period":"2020-05","net_sales_php":"10486809.300000","status":"observed_source_month"},{"period":"2020-06","net_sales_php":"4471548.750000","status":"observed_source_month"},{"period":"2020-07","net_sales_php":"1553987.810000","status":"observed_source_month"},{"period":"2020-08","net_sales_php":"2983110.790000","status":"observed_source_month"},{"period":"2020-09","net_sales_php":"2102633.270000","status":"observed_source_month"},{"period":"2020-10","net_sales_php":"2469548.120000","status":"observed_source_month"},{"period":"2020-11","net_sales_php":"4870933.120000","status":"observed_source_month"},{"period":"2020-12","net_sales_php":"3771013.020000","status":"observed_source_month"},{"period":"2021-01","net_sales_php":"5364380.250000","status":"observed_source_month"},{"period":"2021-02","net_sales_php":"3682482.140000","status":"observed_source_month"},{"period":"2021-03","net_sales_php":"2318360.720000","status":"observed_source_month"},{"period":"2021-04","net_sales_php":"4928899.800000","status":"observed_source_month"},{"period":"2021-05","net_sales_php":"2012916.830000","status":"observed_source_month"},{"period":"2021-06","net_sales_php":"5225498.200000","status":"observed_source_month"},{"period":"2021-07","net_sales_php":"9573660.330000","status":"observed_source_month"},{"period":"2021-08","net_sales_php":"10533739.530000","status":"observed_source_month"},{"period":"2021-09","net_sales_php":"8287059.600000","status":"observed_source_month"},{"period":"2021-10","net_sales_php":"15279603.640000","status":"observed_source_month"},{"period":"2021-11","net_sales_php":"6713804.040000","status":"observed_source_month"},{"period":"2021-12","net_sales_php":"3939801.180000","status":"observed_source_month"},{"period":"2022-01","net_sales_php":"5270833.750000","status":"observed_source_month"},{"period":"2022-02","net_sales_php":"10568697.090000","status":"observed_source_month"},{"period":"2022-03","net_sales_php":"5791537.630000","status":"observed_source_month"},{"period":"2022-04","net_sales_php":"5638889.710000","status":"observed_source_month"},{"period":"2022-05","net_sales_php":"3871579.020000","status":"observed_source_month"},{"period":"2022-06","net_sales_php":"10350221.030000","status":"observed_source_month"},{"period":"2022-07","net_sales_php":"3049922.430000","status":"observed_source_month"},{"period":"2022-08","net_sales_php":"2138525.120000","status":"observed_source_month"},{"period":"2022-09","net_sales_php":"6498970.540000","status":"observed_source_month"},{"period":"2022-10","net_sales_php":"2788157.480000","status":"observed_source_month"},{"period":"2022-11","net_sales_php":"3642191.630000","status":"observed_source_month"},{"period":"2022-12","net_sales_php":"3625464.610000","status":"observed_source_month"},{"period":"2023-01","net_sales_php":"7844551.200000","status":"observed_source_month"},{"period":"2023-02","net_sales_php":"13794078.180000","status":"observed_source_month"},{"period":"2023-03","net_sales_php":"2344746.440000","status":"observed_source_month"},{"period":"2023-04","net_sales_php":"1559602.950000","status":"observed_source_month"},{"period":"2023-05","net_sales_php":"14537368.300000","status":"observed_source_month"},{"period":"2023-06","net_sales_php":"6910935.230000","status":"observed_source_month"},{"period":"2023-07","net_sales_php":"2337078.500000","status":"observed_source_month"},{"period":"2023-08","net_sales_php":"2588026.900000","status":"observed_source_month"},{"period":"2023-09","net_sales_php":"5029660.600000","status":"observed_source_month"},{"period":"2023-10","net_sales_php":"2411760.400000","status":"observed_source_month"},{"period":"2023-11","net_sales_php":"1975407.650000","status":"observed_source_month"},{"period":"2023-12","net_sales_php":"895750.500000","status":"observed_source_month"},{"period":"2024-01","net_sales_php":"2124201.870000","status":"observed_source_month"},{"period":"2024-02","net_sales_php":"1079764.410000","status":"observed_source_month"},{"period":"2024-03","net_sales_php":"2060009.790000","status":"observed_source_month"},{"period":"2024-04","net_sales_php":"1929517.020000","status":"observed_source_month"},{"period":"2024-05","net_sales_php":"1848212.320000","status":"observed_source_month"},{"period":"2024-06","net_sales_php":"6172208.160000","status":"observed_source_month"},{"period":"2024-07","net_sales_php":"2519509.110000","status":"observed_source_month"},{"period":"2024-08","net_sales_php":"2735618.080000","status":"observed_source_month"},{"period":"2024-09","net_sales_php":"1935311.570000","status":"observed_source_month"},{"period":"2024-10","net_sales_php":"1405131.230000","status":"observed_source_month"},{"period":"2024-11","net_sales_php":"1088480.000000","status":"observed_source_month"},{"period":"2024-12","net_sales_php":"3570738.160000","status":"observed_source_month"},{"period":"2025-01","net_sales_php":"2737512.180000","status":"observed_source_month"},{"period":"2025-02","net_sales_php":"8895702.320000","status":"observed_source_month"},{"period":"2025-03","net_sales_php":"1982968.500000","status":"observed_source_month"},{"period":"2025-04","net_sales_php":"4439483.870000","status":"observed_source_month"},{"period":"2025-05","net_sales_php":"7999318.650000","status":"observed_source_month"},{"period":"2025-06","net_sales_php":"12502958.230000","status":"observed_source_month"},{"period":"2025-07","net_sales_php":"3400386.810000","status":"observed_source_month"},{"period":"2025-08","net_sales_php":"6359262.810000","status":"observed_source_month"},{"period":"2025-09","net_sales_php":"16182279.770000","status":"observed_source_month"},{"period":"2025-10","net_sales_php":"7489003.910000","status":"observed_source_month"},{"period":"2025-11","net_sales_php":"6207648.450000","status":"observed_source_month"},{"period":"2025-12","net_sales_php":"10373872.750000","status":"observed_source_month"},{"period":"2026-01","net_sales_php":"2580952.710000","status":"observed_source_month"},{"period":"2026-02","net_sales_php":"17872095.250000","status":"observed_source_month"},{"period":"2026-03","net_sales_php":"8597855.260000","status":"observed_source_month"},{"period":"2026-04","net_sales_php":"19158821.440000","status":"observed_source_month"},{"period":"2026-05","net_sales_php":"9815789.610000","status":"observed_source_month"},{"period":"2026-06","net_sales_php":"15028453.900000","status":"observed_source_month"},{"period":"2026-07","net_sales_php":"24686328.780000","status":"observed_source_month"},{"period":"2026-08","net_sales_php":"16404726.910000","status":"observed_source_month"},{"period":"2026-09","net_sales_php":"26054378.810000","status":"observed_source_month"},{"period":"2026-10","net_sales_php":null,"status":"outside_observed_source_range"},{"period":"2026-11","net_sales_php":null,"status":"outside_observed_source_range"},{"period":"2026-12","net_sales_php":null,"status":"outside_observed_source_range"}],"components":[{"period":"2020-01","observed":"7514695.05","trend":"5515804.577835899","seasonal":"1930837.942788496","residual":"68052.52937560529"},{"period":"2020-02","observed":"4691903","trend":"5385907.44163739","seasonal":"-1082108.3652443646","residual":"388103.9236069741"},{"period":"2020-03","observed":"6637589.08","trend":"5257555.037240971","seasonal":"1736883.1872604142","residual":"-356849.1445013853"},{"period":"2020-04","observed":"1525751.32","trend":"5130638.639882843","seasonal":"1458181.3361822132","residual":"-5063068.656065056"},{"period":"2020-05","observed":"10486809.3","trend":"5005244.980699974","seasonal":"5436192.986707324","residual":"45371.33259270247"},{"period":"2020-06","observed":"4471548.75","trend":"4881569.665530858","seasonal":"-373250.73955554306","residual":"-36770.17597531434"},{"period":"2020-07","observed":"1553987.81","trend":"4759867.006942563","seasonal":"-3086855.02758918","residual":"-119024.16935338266"},{"period":"2020-08","observed":"2983110.79","trend":"4640587.994161387","seasonal":"-1996118.3240567774","residual":"338641.11989539023"},{"period":"2020-09","observed":"2102633.27","trend":"4524618.097504941","seasonal":"-2210321.976317953","residual":"-211662.85118698794"},{"period":"2020-10","observed":"2469548.12","trend":"4413858.110723704","seasonal":"-1710260.5186342692","residual":"-234049.47208943497"},{"period":"2020-11","observed":"4870933.12","trend":"4315190.200742577","seasonal":"1272653.6461682033","residual":"-716910.7269107802"},{"period":"2020-12","observed":"3771013.02","trend":"4254760.97872223","seasonal":"-720804.9323239443","residual":"237056.9736017147"},{"period":"2021-01","observed":"5364380.25","trend":"4209287.841889016","seasonal":"1132591.8023574336","residual":"22500.605753550306"},{"period":"2021-02","observed":"3682482.14","trend":"4189530.797526502","seasonal":"-29111.894119567005","residual":"-477936.76340693515"},{"period":"2021-03","observed":"2318360.72","trend":"4230542.268724137","seasonal":"815953.0188838816","residual":"-2728134.5676080184"},{"period":"2021-04","observed":"4928899.8","trend":"4331128.671872298","seasonal":"816719.3406868415","residual":"-218948.21255913982"},{"period":"2021-05","observed":"2012916.83","trend":"4429615.6272158","seasonal":"3771049.104867525","residual":"-6187747.902083324"},{"period":"2021-06","observed":"5225498.2","trend":"4511292.789742753","seasonal":"727842.4573548391","residual":"-13637.047097591683"},{"period":"2021-07","observed":"9573660.33","trend":"4596790.198767906","seasonal":"-2494485.9446278233","residual":"7471356.075859917"},{"period":"2021-08","observed":"10533739.53","trend":"4689053.488064782","seasonal":"-1732843.8176084314","residual":"7577529.859543649"},{"period":"2021-09","observed":"8287059.6","trend":"4788980.553665002","seasonal":"-258671.25097689749","residual":"3756750.2973118955"},{"period":"2021-10","observed":"15279603.64","trend":"4901588.3119485","seasonal":"-1713434.038379183","residual":"12091449.366430685"},{"period":"2021-11","observed":"6713804.04","trend":"5012611.903630554","seasonal":"300427.140393336","residual":"1400764.99597611"},{"period":"2021-12","observed":"3939801.18","trend":"5102920.592765794","seasonal":"-695301.4363698065","residual":"-467817.9763959879"},{"period":"2022-01","observed":"5270833.75","trend":"5166404.733354847","seasonal":"340087.3722662823","residual":"-235658.35562112927"},{"period":"2022-02","observed":"10568697.09","trend":"5178722.264289621","seasonal":"1031294.4000022371","residual":"4358680.425708141"},{"period":"2022-03","observed":"5791537.63","trend":"5135642.267892517","seasonal":"-113814.5074417532","residual":"769709.8695492363"},{"period":"2022-04","observed":"5638889.71","trend":"5055160.420725272","seasonal":"177700.61425370135","residual":"406028.6750210263"},{"period":"2022-05","observed":"3871579.02","trend":"4969702.4631729135","seasonal":"2114418.4861846855","residual":"-3212541.929357599"},{"period":"2022-06","observed":"10350221.03","trend":"4885786.878206124","seasonal":"1832394.5696485196","residual":"3632039.5821453566"},{"period":"2022-07","observed":"3049922.43","trend":"4796002.983948362","seasonal":"-1906395.758016085","residual":"160315.2040677229"},{"period":"2022-08","observed":"2138525.12","trend":"4691285.396964223","seasonal":"-1457355.1782983274","residual":"-1095405.0986658959"},{"period":"2022-09","observed":"6498970.54","trend":"4561413.723298751","seasonal":"1687056.2832463363","residual":"250500.53345491271"},{"period":"2022-10","observed":"2788157.48","trend":"4415863.860097845","seasonal":"-1730600.7652069477","residual":"102894.38510910235"},{"period":"2022-11","observed":"3642191.63","trend":"4288963.028065166","seasonal":"-676583.751765005","residual":"29812.353699838743"},{"period":"2022-12","observed":"3625464.61","trend":"4189982.9355588118","seasonal":"-665884.941175987","residual":"101366.61561717559"},{"period":"2023-01","observed":"7844551.2","trend":"4110817.306215787","seasonal":"-450218.64664492646","residual":"4183952.54042914"},{"period":"2023-02","observed":"13794078.18","trend":"4049531.686845008","seasonal":"2092403.1323290209","residual":"7652143.360825971"},{"period":"2023-03","observed":"2344746.44","trend":"3996390.541272422","seasonal":"-1043321.0125315029","residual":"-608323.0887409193"},{"period":"2023-04","observed":"1559602.95","trend":"3940721.33350062","seasonal":"-452967.1260713592","residual":"-1928151.2574292608"},{"period":"2023-05","observed":"14537368.3","trend":"3878534.188825804","seasonal":"460732.6969700014","residual":"10198101.414204195"},{"period":"2023-06","observed":"6910935.23","trend":"3812057.1461491967","seasonal":"2941415.248915416","residual":"157462.8349353876"},{"period":"2023-07","observed":"2337078.5","trend":"3757934.038112932","seasonal":"-1325347.3598563753","residual":"-95508.17825655639"},{"period":"2023-08","observed":"2588026.9","trend":"3723470.9057761305","seasonal":"-1181657.1351260329","residual":"46213.12934990227"},{"period":"2023-09","observed":"5029660.6","trend":"3700234.3746126792","seasonal":"3635591.151246994","residual":"-2306164.9258596734"},{"period":"2023-10","observed":"2411760.4","trend":"3669056.8836926087","seasonal":"-1768949.1198427756","residual":"511652.63615016686"},{"period":"2023-11","observed":"1975407.65","trend":"3624236.7919327226","seasonal":"-1683690.9224319065","residual":"34861.78049918357"},{"period":"2023-12","observed":"895750.5","trend":"3572091.0621198276","seasonal":"-618903.0443107614","residual":"-2057437.5178090662"},{"period":"2024-01","observed":"2124201.87","trend":"3515362.8325587716","seasonal":"-1231458.9999795444","residual":"-159701.96257922705"},{"period":"2024-02","observed":"1079764.41","trend":"3461984.099350234","seasonal":"3215197.2599720717","residual":"-5597416.949322306"},{"period":"2024-03","observed":"2060009.79","trend":"3420565.482235063","seasonal":"-2036886.547299262","residual":"676330.8550641993"},{"period":"2024-04","observed":"1929517.02","trend":"3397154.9384627887","seasonal":"-1086143.0272946218","residual":"-381494.8911681669"},{"period":"2024-05","observed":"1848212.32","trend":"3397732.8810217744","seasonal":"-1158540.7364834757","residual":"-390979.8245382989"},{"period":"2024-06","observed":"6172208.16","trend":"3426163.052921415","seasonal":"4058303.282164801","residual":"-1312258.1750862156"},{"period":"2024-07","observed":"2519509.11","trend":"3485968.859973727","seasonal":"-766703.1929554823","residual":"-199756.55701824464"},{"period":"2024-08","observed":"2735618.08","trend":"3588446.424058762","seasonal":"-822313.9230378997","residual":"-30514.42102086218"},{"period":"2024-09","observed":"1935311.57","trend":"3733302.0311279176","seasonal":"5552646.11042282","residual":"-7350636.571550737"},{"period":"2024-10","observed":"1405131.23","trend":"3921503.395751697","seasonal":"-1833910.6650474148","residual":"-682461.500704282"},{"period":"2024-11","observed":"1088480","trend":"4154206.1663902374","seasonal":"-2734006.1292612543","residual":"-331720.03712898307"},{"period":"2024-12","observed":"3570738.16","trend":"4431188.794315551","seasonal":"-552558.1131934405","residual":"-307892.521122111"},{"period":"2025-01","observed":"2737512.18","trend":"4766839.068796109","seasonal":"-2005808.757206929","residual":"-23518.13158917986"},{"period":"2025-02","observed":"8895702.32","trend":"5163894.742677662","seasonal":"4341761.594507316","residual":"-609954.0171849774"},{"period":"2025-03","observed":"1982968.5","trend":"5610932.061270036","seasonal":"-3042731.348975033","residual":"-585232.2122950032"},{"period":"2025-04","observed":"4439483.87","trend":"6093860.4724081475","seasonal":"-1720467.0707896275","residual":"66090.46838148031"},{"period":"2025-05","observed":"7999318.65","trend":"6609190.851236182","seasonal":"-2772017.978915665","residual":"4162145.7776794834"},{"period":"2025-06","observed":"12502958.23","trend":"7156346.789112934","seasonal":"5173845.893492644","residual":"172765.5473944228"},{"period":"2025-07","observed":"3400386.81","trend":"7723811.076504466","seasonal":"-218509.12552465603","residual":"-4104915.14097981"},{"period":"2025-08","observed":"6359262.81","trend":"8306845.454547976","seasonal":"-434560.04565957777","residual":"-1513022.598888399"},{"period":"2025-09","observed":"16182279.77","trend":"8906750.36484304","seasonal":"7464134.571313762","residual":"-188605.16615680233"},{"period":"2025-10","observed":"7489003.91","trend":"9521055.45892559","seasonal":"-1918854.3589326623","residual":"-113197.18999292701"},{"period":"2025-11","observed":"6207648.45","trend":"10149710.75670004","seasonal":"-3801866.2197648403","residual":"-140196.0869351998"},{"period":"2025-12","observed":"10373872.75","trend":"10815637.697555615","seasonal":"-479151.86093155644","residual":"37386.91337594204"},{"period":"2026-01","observed":"2580952.71","trend":"11485835.394631559","seasonal":"-2771375.3169003916","residual":"-6133507.367731167"},{"period":"2026-02","observed":"17872095.25","trend":"12156300.846732978","seasonal":"5476648.429145071","residual":"239145.97412195057"},{"period":"2026-03","observed":"8597855.26","trend":"12827716.209361752","seasonal":"-4063671.3859773264","residual":"-166189.56338442676"},{"period":"2026-04","observed":"19158821.44","trend":"13500161.154028961","seasonal":"-2349543.902237931","residual":"8008204.188208969"},{"period":"2026-05","observed":"9815789.61","trend":"14173605.64630623","seasonal":"-4371800.261676029","residual":"13984.22536979802"},{"period":"2026-06","observed":"15028453.9","trend":"14847947.695939794","seasonal":"6295654.299371059","residual":"-6115148.095310852"},{"period":"2026-07","observed":"24686328.78","trend":"15523041.271705637","seasonal":"319623.65950578696","residual":"8843663.848788576"},{"period":"2026-08","observed":"16404726.91","trend":"16198740.949303228","seasonal":"-29723.677875700756","residual":"235709.63857247308"},{"period":"2026-09","observed":"26054378.81","trend":"16874852.20382712","seasonal":"9367202.454488523","residual":"-187675.848315645"}],"contracts_excluded":true,"financial_release_approved":false,"model_publication_approved":false}')
PACKAGE_SHA256 = '05e9c3c65268387a93bb136b71a0b85dd50ed61acf45184d4e3f64b829d0ac13'
VERIFIED_EXPORT_HASHES = {'calendar': 'dd38038f620a8aa3fec09aa2d40163ec1e7dbef717e405c02dc867a0941951fd', 'components': 'd47763d7a195a4dad1d731869ba18b369e90b03195025e2536651a1baf4328e8'}

"""Generated into one self-contained staging notebook; no application serving changes."""
import base64
import hashlib
import json
from decimal import Decimal

CANDIDATE_SCHEMA = 'workspace.medshield_original_stl_20261027'
CANDIDATE_COLUMNS = ['row_id', 'record_type', 'period', 'release_id', 'dataset_id',
                     'policy_version', 'payload_sha256', 'payload_json']


def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'), allow_nan=False)


def content_hash(value):
    return hashlib.sha256(canonical_json(value).encode('utf-8')).hexdigest()


def checked_package(package, expected_hash):
    require(content_hash(package) == expected_hash, 'STL package bytes differ from reviewed export package')
    source = package['source']
    require(source == {'source_table': SOURCE_TABLE, 'source_delta_version': 0,
                       'dataset_id': DATASET_ID, 'policy': POLICY,
                       'receipt_sha256': RECEIPT_SHA256, 'monthly_sha256': MONTHLY_LOGICAL_SHA256},
            'STL package source identity differs')
    require(package['policy'] == 'original_itemized_stl_export_candidate_v1', 'Unexpected STL candidate policy')
    monthly = validate_monthly(package['monthly_rows'])
    require(package['calendar'] == calendar_rows(monthly), 'Calendar differs from checked monthly source')
    require(package['method'] == {'name': 'STL', 'period': 12, 'seasonal_window': 13, 'robust': True,
                                 'minimum_contiguous_months': 36, 'missing_policy': 'no_imputation_split_at_gaps',
                                 'statsmodels_version': '0.14.6', 'numpy_version': '2.3.4'}, 'STL method differs')
    require(package['contracts_excluded'] is True and package['financial_release_approved'] is False
            and package['model_publication_approved'] is False, 'Unexpected scope or approval')
    require(package['source_export_hashes'] == VERIFIED_EXPORT_HASHES, 'STL exported-evidence identities differ')
    calendar = package['calendar']
    runs, current = [], []
    for row in calendar:
        if row['status'] == 'observed_source_month':
            current.append(row['period'])
        elif current:
            runs.append(current)
            current = []
    if current:
        runs.append(current)
    expected_periods = [period for run in runs if len(run) >= 36 for period in run]
    components = package['components']
    require([row['period'] for row in components] == expected_periods, 'STL component coverage differs')
    require(len(calendar) == 120 and len(components) == 81, 'Unexpected STL calendar/component count')
    observed = {row['period']: row['net_sales_php'] for row in calendar}
    for row in components:
        require(set(row) == {'period', 'observed', 'trend', 'seasonal', 'residual'}, 'Unexpected component fields')
        values = {key: numeric(row[key]) for key in ('observed', 'trend', 'seasonal', 'residual')}
        require(values['observed'] == numeric(observed[row['period']]), 'STL observation differs from Gold monthly input')
        require(abs(values['observed'] - values['trend'] - values['seasonal'] - values['residual']) <= Decimal('.000001'),
                'STL export reconstruction differs')
    return monthly


def candidate_records(package, expected_hash):
    checked_package(package, expected_hash)
    release = 'stl_' + expected_hash[:24]
    metadata = {'release_id': release, 'package_sha256': expected_hash, **package['source'], 'source': package['source'],
                'policy': package['policy'], 'method': package['method'], 'controls': EXPECTED_TOTALS,
                'source_export_hashes': package['source_export_hashes'],
                'population': 'original_itemized_sales_contracts_excluded', 'contracts_excluded': True,
                'financial_release_approved': False, 'model_publication_approved': False,
                'retrospective_diagnostic_only': True, 'origin_safe_forecast_feature': False,
                'calendar_records': 120, 'component_records': 81,
                'limitations': ['Not complete company sales or verified medicine demand.',
                                'No cross-gap imputation; short observed segments have no fitted components.',
                                'Full-series retrospective STL is not a forecast or an origin-safe predictor.'],
                'evidence_origin': 'verified_operator_Databricks_STL_exports',
                'approved_for_live_serving': False}
    rows = []
    for kind, values in [('calendar', package['calendar']), ('component', package['components']), ('receipt', [metadata])]:
        for value in values:
            period = value.get('period')
            payload = canonical_json(value)
            rows.append({'row_id': kind + ':' + (period or 'metadata'), 'record_type': kind,
                         'period': period, 'release_id': release, 'dataset_id': DATASET_ID,
                         'policy_version': package['policy'],
                         'payload_sha256': hashlib.sha256(payload.encode('utf-8')).hexdigest(), 'payload_json': payload})
    require(len(rows) == 202 and len({row['row_id'] for row in rows}) == 202, 'Duplicate/missing candidate record')
    return sorted(rows, key=lambda row: row['row_id'])


def sql_text(value):
    if value is None:
        return 'CAST(NULL AS STRING)'
    # Base64 avoids SQL quote/backslash interpretation inside JSON/text values.
    encoded = base64.b64encode(value.encode('utf-8')).decode('ascii')
    return "CAST(unbase64('" + encoded + "') AS STRING)"


def create_candidate_sql(table, rows):
    require(table == CANDIDATE_SCHEMA + '.stl_' + PACKAGE_SHA256[:24] + '_candidate', 'Candidate target differs')
    require(len(rows) == 202, 'Candidate row bound differs')
    values = ',\n'.join('(' + ','.join(sql_text(row[column]) for column in CANDIDATE_COLUMNS) + ')' for row in rows)
    statement = 'CREATE TABLE ' + table + ' USING DELTA AS SELECT * FROM VALUES\n' + values + '\nAS candidate (' + ','.join(CANDIDATE_COLUMNS) + ')'
    require(len(statement.encode('utf-8')) < 2_000_000, 'Candidate statement exceeds safety limit')
    return statement


def verify_candidate_readback(session, table, expected):
    history = session.sql('DESCRIBE HISTORY ' + table + ' LIMIT 1').collect()
    require(len(history) == 1 and int(history[0]['version']) == 0, 'Candidate table is no longer immutable Delta0')
    actual = session.sql('SELECT ' + ','.join(CANDIDATE_COLUMNS) + ' FROM ' + table
                         + ' VERSION AS OF 0 ORDER BY row_id LIMIT 203').collect()
    rows = [row.asDict() for row in actual]
    require(rows == expected, 'Candidate readback differs; do not overwrite or activate')
    for row in rows:
        require(hashlib.sha256(row['payload_json'].encode('utf-8')).hexdigest() == row['payload_sha256'],
                'Candidate payload checksum differs')
    return content_hash(rows)


def stage_original_stl_candidate(session):
    # All source/package/coverage checks finish before the first DDL statement.
    monthly = read_gold_months(session)
    expected_monthly = checked_package(PACKAGE, PACKAGE_SHA256)
    require(monthly_identity(monthly) == monthly_identity(expected_monthly), 'Live Gold differs from staged evidence')
    rows = candidate_records(PACKAGE, PACKAGE_SHA256)
    release = rows[0]['release_id']
    table = CANDIDATE_SCHEMA + '.' + release + '_candidate'
    created = False
    # CREATE has no replace/overwrite/append/IF NOT EXISTS bypass. A concurrent
    # create may fail; retry will verify the existing content without rewriting.
    if not session.catalog.tableExists(table):
        session.sql('CREATE SCHEMA IF NOT EXISTS ' + CANDIDATE_SCHEMA)
        session.sql(create_candidate_sql(table, rows))
        created = True
    readback_hash = verify_candidate_readback(session, table, rows)
    return {'status': 'ORIGINAL_ITEMIZED_STL_CANDIDATE_STAGED', 'release_id': release,
            'candidate_table': table, 'candidate_delta_version': 0, 'package_sha256': PACKAGE_SHA256,
            'records_sha256': readback_hash, 'source': PACKAGE['source'],
            'calendar_records': 120, 'component_records': 81, 'receipt_records': 1,
            'new_candidate_tables_created': int(created), 'identical_existing_candidate_verified': not created,
            'source_files_modified': False, 'source_tables_modified': False, 'live_serving_changed': False,
            'category_approvals_added': 0, 'financial_release_approved': False,
            'model_publication_approved': False, 'forecasts_trained': 0}


## Create candidate and verify exact readback
Run this cell to create the isolated candidate or verify its identical existing Delta0 contents. A mismatch stops without an overwrite. Keep the receipt output for the next gateway integration step.


In [ ]:
_stl_candidate_receipt = stage_original_stl_candidate(spark)
print(json.dumps(_stl_candidate_receipt, indent=2))
print('PASS_ORIGINAL_ITEMIZED_STL_CANDIDATE. Exact Delta0 readback verified; live serving unchanged.')
